In [1]:
import sqlite3
import pandas as pd
import logging
from ingestion_db import ingest_db

logging.basicConfig(
    filename = "logs/get_vendor_summary.log",
    level = logging.DEBUG,
    format = "%(asctime)s - %(levelname)s - %(message)s",
    filemode = "a"
)

def create_vendor_summary(conn):
    #This function will merge the different tables to get the overall vendor summary and adding new columns
    Vendor_Sales_Summary = pd.read_sql_query("""WITH FreightSummary AS (
        SELECT
            VendorNumber, 
            SUM(Freight) AS FreightCost
        FROM vendor_invoice
        GROUP BY VendorNumber
    ),

    PurchaseSummary AS (
        SELECT
            p.VendorNumber, 
            p.VendorName, 
            p.Brand,
            p.Description,
            p.PurchasePrice,
            pp.Volume,
            pp.Price as ActualPrice,
            SUM(p.Quantity) as TotalPurchaseQuantity, 
            SUM(p.Dollars) as TotalPurchaseDollars 
        FROM purchases p
        JOIN purchase_prices pp
            ON p.Brand = pp.Brand
        WHERE p.PurchasePrice > 0
        GROUP BY p.VendorNumber, p.VendorName, p.Brand, p.Description, p.PurchasePrice, pp.Price, pp.Volume
    ),

    SalesSummary AS (
        SELECT
            VendorNo,
            Brand,
            SUM(SalesPrice) AS TotalSalesPrice,
            SUM(SalesDollars) AS TotalSalesDollars,
            SUM(SalesQuantity) AS TotalSalesQuantity,
            SUM(ExciseTax) AS TotalExciseTax
        FROM sales
        GROUP BY VendorNo, Brand
    )

    SELECT
        ps.VendorNumber, 
        ps.VendorName,
        ps.Brand,
        ps.Description,
        ps.PurchasePrice,
        ps.ActualPrice,
        ps.Volume,
        ps.TotalPurchaseQuantity,
        ps.TotalPurchaseDollars,
        ss.TotalSalesPrice,
        ss.TotalSalesDollars,
        ss.TotalSalesQuantity,
        ss.TotalExciseTax,
        fs.FreightCost
    FROM PurchaseSummary ps
    LEFT JOIN SalesSummary ss
        ON ss.VendorNo = ps.VendorNumber
        AND ss.Brand = ps.Brand
    LEFT JOIN FreightSummary fs
        ON fs.VendorNumber = ps.VendorNumber
    ORDER BY ps.TotalPurchaseDollars DESC""", conn)
    
    return Vendor_Sales_Summary


def clean_data(Vendor_Sales_Summary):
    # Changing the datatype to float
    Vendor_Sales_Summary['Volume'] = Vendor_Sales_Summary['Volume'].astype('float64')
    
    # Filling the null values with 0
    Vendor_Sales_Summary.fillna(0, inplace = True)
    
    # Removing the Extra Spaces
    Vendor_Sales_Summary['VendorName'] = Vendor_Sales_Summary['VendorName'].str.strip()
    Vendor_Sales_Summary['Description'] = Vendor_Sales_Summary['Description'].str.strip()

    # Creating some extra features for better understanding
    Vendor_Sales_Summary['GrossProfit'] = Vendor_Sales_Summary['TotalSalesDollars'] - Vendor_Sales_Summary['TotalPurchaseDollars']
    Vendor_Sales_Summary['ProfitMargin'] = (Vendor_Sales_Summary['GrossProfit']/Vendor_Sales_Summary['TotalSalesDollars'])*100
    Vendor_Sales_Summary['StockTurnover'] = (Vendor_Sales_Summary['TotalSalesQuantity']/Vendor_Sales_Summary['TotalPurchaseQuantity'])
    Vendor_Sales_Summary['SalestoPurchaseRatio'] = (Vendor_Sales_Summary['TotalSalesDollars']/Vendor_Sales_Summary['TotalPurchaseDollars'])

    return Vendor_Sales_Summary
    
    
if __name__ == '__main__':
    
    # Creating database connections
    conn = sqlite3.connect('inventory.db')
    
    logging.info('Creating Vendor Summary Table......')
    summary_df = create_vendor_summary(conn)
    logging.info(summary_df.head())
    
    logging.info('Cleaning Data......')
    Clean_df = clean_data(summary_df)
    logging.info(Clean_df.head())
    
    logging.info('Ingesting Data......')
    ingest_db(Clean_df, 'Vendor_Sales_Summary', conn)
    logging.info('Completed')
    
    